<a href="https://colab.research.google.com/github/Student-Abhishekkumar/15-C-plus-plus-programs-for-practice./blob/main/Voxcpm_Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ── Setup: run this first every session ───────────────────────────────────
!pip install voxcpm soundfile -q

import warnings, re, os
import numpy as np
warnings.filterwarnings("ignore")

from voxcpm import VoxCPM
import soundfile as sf

model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
print("✅ Ready! Now run the next cell.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 19.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 10.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 31.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.6/19.6 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.0/812.0 kB 55.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

voxcpm_model_path: /root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/bffb3df5a29440629464e5e839f4d214c8714c3d, zipenhancer_model_path: None, enable_denoiser: False
Loading AudioVAE from pytorch: /root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/bffb3df5a29440629464e5e839f4d214c8714c3d/audiovae.pth
Running on device: cuda, dtype: bfloat16
Loading model from safetensors: /root/.cache/huggingface/hub/models--openbmb--VoxCPM2/snapshots/bffb3df5a29440629464e5e839f4d214c8714c3d/model.safetensors
Loaded VoxCPM2Model
Warm up VoxCPMModel...
W0418 16:49:34.472000 1208 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode
100%|██████████| 10/10 [00:50<00:00,  5.08s/it]


✅ Ready! Now run the next cell.


In [ ]:
# ── Full restart: upload → read → split → voice → generate → stitch ───────
import re, os, warnings
import soundfile as sf
import numpy as np
from google.colab import files
from IPython.display import Audio
warnings.filterwarnings("ignore")

# 1. Upload
print("Upload your .txt file:")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# 2. Read
with open(filename, 'r', encoding='utf-8') as f:
    full_text = f.read()
print(f"✓ {len(full_text)} characters, ~{len(full_text.split())} words")

# 3. Split
def split_into_chunks(text, max_chars=2500):
    sentences = re.split(r'(?<=[.!?])\s+', text)
    chunks = []
    current = ""
    for sentence in sentences:
        if not sentence.strip():
            continue
        if len(current) + len(sentence) + 1 <= max_chars:
            current += sentence + " "
        else:
            if current.strip():
                chunks.append(current.strip())
            current = sentence + " "
    if current.strip():
        chunks.append(current.strip())
    return chunks

chunks = split_into_chunks(full_text, max_chars=2500)
print(f"✓ Split into {len(chunks)} chunks")

# 4. Voice — change "Dread" to any: Echo, Nova, Sage, Luna, Blaze
voices = {
    "Dread":  "(Very deep, intimidating, slow, heavy voice, villain-like presence)"
}
voice_prompt = voices["Dread"]  # ← change here
print(f"✓ Voice: {voice_prompt}")

# 5. Generate chunks
output_dir = "audio_chunks"
os.makedirs(output_dir, exist_ok=True)
saved_files = []
failed_chunks = []

print(f"\n🎙️  Generating {len(chunks)} chunks...\n")
for i, chunk in enumerate(chunks):
    chunk_num = i + 1
    print(f"Chunk {chunk_num}/{len(chunks)} — {len(chunk)} chars")
    print(f"  {chunk[:80]}...")
    try:
        wav = model.generate(
            text=f"{voice_prompt}{chunk}",
            cfg_value=2.0,
            inference_timesteps=10,
        )
        out_path = f"{output_dir}/{chunk_num}.wav"
        sf.write(out_path, wav, model.tts_model.sample_rate)
        duration = round(len(wav) / model.tts_model.sample_rate, 1)
        saved_files.append(out_path)
        print(f"  ✓ {duration}s saved\n")
    except Exception as e:
        print(f"  ✗ Failed: {e} — skipping\n")
        failed_chunks.append(chunk_num)

print(f"✅ Generated {len(saved_files)}/{len(chunks)} chunks")
if failed_chunks:
    print(f"⚠️  Failed chunks: {failed_chunks}")

# 6. Stitch
print("\n🔗 Stitching all chunks...")
all_audio = []
silence = np.zeros(int(model.tts_model.sample_rate * 0.4))
for path in saved_files:
    wav, _ = sf.read(path)
    all_audio.append(wav)
    all_audio.append(silence)

full_audio = np.concatenate(all_audio)
output_name = filename.replace('.txt', '_narration.wav').replace(' ', '_')
sf.write(output_name, full_audio, model.tts_model.sample_rate)

mins = len(full_audio) // model.tts_model.sample_rate // 60
secs = (len(full_audio) // model.tts_model.sample_rate) % 60
print(f"✓ Total: {mins}m {secs}s → {output_name}")

# 7. Download
files.download(output_name)
print("✅ Done! Check your downloads folder.")

Upload your .txt file:


Saving When Aster was dragged to the Elmhe.txt to When Aster was dragged to the Elmhe.txt
✓ 5483 characters, ~987 words
✓ Split into 3 chunks
✓ Voice: (Very deep, intimidating, slow, heavy voice, villain-like presence)

🎙️  Generating 3 chunks...

Chunk 1/3 — 2436 chars
  Mega Manhwa Explainer: The Revenge of the 7-Winged Heir –
(Opening Hook) Imagine...


 12%|█▏        | 509/4096 [03:12<22:34,  2.65it/s]


  ✓ 81.6s saved

Chunk 2/3 — 2449 chars
  Use House Hifried ke head milte hain jo use adopt kar lete hain. Aster dekhta ha...


  2%|▏         | 73/4096 [00:28<25:56,  2.58it/s]


...

 14%|█▍        | 195/1378 [01:13<07:28,  2.64it/s]


  ✓ 31.4s saved

✅ Generated 3/3 chunks

🔗 Stitching all chunks...
✓ Total: 2m 6s → When_Aster_was_dragged_to_the_Elmhe_narration.wav


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done! Check your downloads folder.
